# FF100 — Covariance and Log-Euclidean SPD ablation

Dataset: **`ken_french_100_size_bm`**, **100 assets**. This transfers an existing KF12 experiment;
it does not assume KF12 outcomes or tune new methods. **Exploratory / scalability only:** the complete-case FF100 panel has multi-year gaps and only **68 validation windows** after segmentation. Do not infer robust forecasting superiority.

The saved `configs/cde_ff100.yaml` protocol remains unchanged: requested sample **1958–2023**,
training through **1999-12-31**, validation **2000–2004**, lookback **60**, horizon **1**, seed **42**.
Arithmetic decimal returns minus daily French RF; complete-case row exclusion without imputation;
seven-day calendar-gap segmentation and train-only standardization. **No test evaluation.**

**Prerequisite:** completed, consistent seed-42 CDE at `/kaggle/working/ff100_cde`
.
The CDE stays frozen; extracted train/validation features are cached.
Finish those runs in their own notebooks; this notebook never trains a missing reference.

Work through individual cells. Extraction/training/evaluation and the final upload are manual steps;
**Run All can be expensive and uploads artifacts**. Outputs are isolated at `/kaggle/working/ff100_spd` and
`high_dimensional/ff100_spd` in the central Kaggle dataset. No experiment results are populated here.

## Setup

Enable Internet and a GPU in Kaggle settings. This is the established KF12 GitHub sync cell;
`BRANCH` selects the checkout containing these notebooks. Use `main` after merging.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/high-dimensional-kaggle"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev,cde]'

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Lightweight regression tests

In [ ]:
!pytest -q tests/semantic tests/unit/test_multiseed.py tests/unit/test_evidential_regimes.py \
    tests/unit/test_routing.py tests/unit/test_spd_features.py tests/unit/test_topology.py \
    tests/unit/test_precomputed_conditioning.py tests/unit/test_highdim_research.py
assert _exit_code == 0, "Regression tests failed."

## Restore prerequisites and this experiment

Only the listed reference runs and this notebook's own namespace are restored. Local folders
are retained. Missing references fail the next check; there is no KF12 fallback.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import torch
import kagglehub
from diffusion_portfolio.utils.research_checks import (
    check_checkpoint_pair, require_completed_run, source_identity,
)

HANDLE = "amirentezari/diffusion-portfolio"
EXPERIMENT = "high_dimensional/ff100_spd"
ROOT = Path("/kaggle/working/ff100_spd")
CDE_DIR = Path("/kaggle/working/ff100_cde")
REFERENCES = {Path(path): config for path, config in {'/kaggle/working/ff100_cde': 'configs/cde_ff100.yaml'}.items()}

In [ ]:
artifact_path = Path(kagglehub.dataset_download(HANDLE))
restores = {"high_dimensional/" + path.name: path for path in REFERENCES}
restores[EXPERIMENT] = ROOT
for relative, destination in restores.items():
    source = artifact_path / relative
    if destination.exists():
        print("Already available:", destination)
    elif source.is_dir():
        if destination in REFERENCES:
            require_completed_run(source, REFERENCES[destination], assets=100, validation=False)
        shutil.copytree(source, destination)
        print("Restored:", destination)
    else:
        print("No backup:", relative)

In [ ]:
for path, config in REFERENCES.items():
    require_completed_run(path, config, assets=100, validation=False)
cfg = require_completed_run(CDE_DIR, "configs/cde_ff100.yaml", assets=100)
SOURCE = source_identity(CDE_DIR)
print("Prerequisites complete:", cfg.data.dataset, "assets:", 100, "seed:", cfg.seed)

## Feasibility before full extraction

Check **two training histories** with the real frozen encoder and original feature functions.
The report includes dimensions, finite-value checks, usable split counts, and an approximate
dense cache size. It does not write a full cache. Inspect GPU/RAM availability before extraction.
Covariance uses the original sample estimator, relative eigenvalue floor **1e-6**, matrix log and isometric upper-triangle vectorization: **5050 features** per representation. With lookback 60, FF100 sample covariance has rank at most 59. The floor is essential; a non-positive reconstructed eigenvalue or non-finite log stops this check. No floor/dtype/dimension reduction is substituted. Constant columns are retained as in KF12.

Peak CPU RAM exceeds the cache estimate: return windows, raw/scaled features, and extraction
concatenations coexist. This is a feasibility gate, not an assumption that a T4 can fit every batch.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/extract_features.py \
    --kind spd \
    --cde-run-dir /kaggle/working/ff100_cde \
    --output-dir /kaggle/working/ff100_spd/features \
    --batch-size 2 \
    --eigenvalue-floor 1e-6 \
    --device cuda \
    --smoke
assert _exit_code == 0, "feature-smoke failed."

## Extract and cache train/validation features

This is expensive. The encoder is frozen and evaluated without gradients. `--reuse` validates
source identity, exact prepared returns/dates, finite values, and train-only scaling before reusing a complete cache.
A partial/mismatched cache is rejected; it is never silently deleted or overwritten.
Full extraction uses the original batch size. A smaller extraction batch can be explicitly
chosen if memory requires it; do not change the CDE solver or feature definitions.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/extract_features.py \
    --kind spd \
    --cde-run-dir /kaggle/working/ff100_cde \
    --output-dir /kaggle/working/ff100_spd/features \
    --batch-size 128 \
    --eigenvalue-floor 1e-6 \
    --device cuda \
    --reuse
assert _exit_code == 0, "extract failed."

In [ ]:
from diffusion_portfolio.data.preparation import prepare_return_data
from diffusion_portfolio.data.precompute import validate_feature_cache

datasets, scaler = prepare_return_data(cfg, standardizer_path=CDE_DIR / "standardizer.npz")
metadata = validate_feature_cache(ROOT / "features", datasets, 'spd')
assert metadata["source"] == SOURCE
display(pd.Series({k: v for k, v in metadata.items() if k.endswith("_dim")}, name="Feature dimensions"))
del datasets  # Free dense return windows before training.

## Train cde control/ablation

Uses cached features, the unchanged 50-epoch configuration, and the shared one-line-per-epoch trainer. Resume is supported.

In [ ]:
check_checkpoint_pair(ROOT / "cde")

!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant cde \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --output-dir /kaggle/working/ff100_spd/cde \
    --device cuda \
    --resume
assert _exit_code == 0, "train-cde failed."

In [ ]:
RUN_DIR = ROOT / "cde"

import pandas as pd
import matplotlib.pyplot as plt

# last.pt contains the most recent epoch history even if a run was interrupted.
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(
        torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"]
    )
else:
    history = pd.read_csv(RUN_DIR / "history.csv")
display(history)
best = history.loc[history["val_loss"].idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train Loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation Loss")
ax.scatter(best["epoch"], best["val_loss"], color="black", zorder=3,
           label=f"Best validation: epoch {int(best['epoch'])}")
ax.set(xlabel="Epoch", ylabel="Diffusion loss", title="FF100 — cde conditioning")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f"Best validation epoch: {int(best['epoch'])}; loss: {best['val_loss']:.6f}")

### Best-validation checkpoint: cde

In [ ]:
require_completed_run(ROOT / "cde", assets=100)

!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant cde \
    --run-dir /kaggle/working/ff100_spd/cde \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --split validation \
    --batch-size 32 \
    --device cuda
assert _exit_code == 0, "evaluate-cde failed."

## Train covariance control/ablation

Uses cached features, the unchanged 50-epoch configuration, and the shared one-line-per-epoch trainer. Resume is supported.

In [ ]:
check_checkpoint_pair(ROOT / "covariance")

!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant covariance \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --output-dir /kaggle/working/ff100_spd/covariance \
    --device cuda \
    --resume
assert _exit_code == 0, "train-covariance failed."

In [ ]:
RUN_DIR = ROOT / "covariance"

import pandas as pd
import matplotlib.pyplot as plt

# last.pt contains the most recent epoch history even if a run was interrupted.
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(
        torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"]
    )
else:
    history = pd.read_csv(RUN_DIR / "history.csv")
display(history)
best = history.loc[history["val_loss"].idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train Loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation Loss")
ax.scatter(best["epoch"], best["val_loss"], color="black", zorder=3,
           label=f"Best validation: epoch {int(best['epoch'])}")
ax.set(xlabel="Epoch", ylabel="Diffusion loss", title="FF100 — covariance conditioning")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f"Best validation epoch: {int(best['epoch'])}; loss: {best['val_loss']:.6f}")

### Best-validation checkpoint: covariance

In [ ]:
require_completed_run(ROOT / "covariance", assets=100)

!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant covariance \
    --run-dir /kaggle/working/ff100_spd/covariance \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --split validation \
    --batch-size 32 \
    --device cuda
assert _exit_code == 0, "evaluate-covariance failed."

## Train log_spd control/ablation

Uses cached features, the unchanged 50-epoch configuration, and the shared one-line-per-epoch trainer. Resume is supported.

In [ ]:
check_checkpoint_pair(ROOT / "log_spd")

!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant log_spd \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --output-dir /kaggle/working/ff100_spd/log_spd \
    --device cuda \
    --resume
assert _exit_code == 0, "train-log_spd failed."

In [ ]:
RUN_DIR = ROOT / "log_spd"

import pandas as pd
import matplotlib.pyplot as plt

# last.pt contains the most recent epoch history even if a run was interrupted.
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(
        torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"]
    )
else:
    history = pd.read_csv(RUN_DIR / "history.csv")
display(history)
best = history.loc[history["val_loss"].idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train Loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation Loss")
ax.scatter(best["epoch"], best["val_loss"], color="black", zorder=3,
           label=f"Best validation: epoch {int(best['epoch'])}")
ax.set(xlabel="Epoch", ylabel="Diffusion loss", title="FF100 — log_spd conditioning")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f"Best validation epoch: {int(best['epoch'])}; loss: {best['val_loss']:.6f}")

### Best-validation checkpoint: log_spd

In [ ]:
require_completed_run(ROOT / "log_spd", assets=100)

!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant log_spd \
    --run-dir /kaggle/working/ff100_spd/log_spd \
    --cde-run-dir /kaggle/working/ff100_cde \
    --feature-dir /kaggle/working/ff100_spd/features \
    --split validation \
    --batch-size 32 \
    --device cuda
assert _exit_code == 0, "evaluate-log_spd failed."

## Completed validation comparison

These are newly trained controlled ablations, not the original CDE checkpoint's historical loss.
CRPS, Energy Score, and calibration come from each variant's best-validation checkpoint.
Compare TDA against geometry, or Log-SPD against ordinary covariance, before making a specific claim.

In [ ]:
rows, calibration = [], []
for variant in ('cde', 'covariance', 'log_spd'):
    run = ROOT / variant
    require_completed_run(run, assets=100, validation=True)
    training = json.loads((run / "summary.json").read_text())
    assert training["source"] == SOURCE and training["variant"] == variant
    metrics = json.loads((run / "validation_probabilistic_metrics.json").read_text())
    rows.append({"Variant": variant, "Best epoch": training["best_epoch"],
                 "Validation loss": training["best_val_loss"], "CRPS": metrics["crps_mean"],
                 "Energy Score": metrics["energy_score"], "Parameters": training["parameter_count"],
                 "Mean |ACE|": pd.DataFrame(metrics["calibration"])["ace"].abs().mean()})
    calibration.append(pd.DataFrame(metrics["calibration"]).assign(Variant=variant))
display(pd.DataFrame(rows).set_index("Variant"))
display(pd.concat(calibration, ignore_index=True))

## Manual backup

Click only when ready to upload. The latest central dataset is copied in full, then only this
notebook's own namespace is replaced. **Serialize uploads across Kaggle sessions**; there is no
cross-session lock or automatic merge. A failed/empty download stops upload. Prerequisite seed-42
directories are never replaced here. Resumable diffusion checkpoints can be backed up mid-run;
probe/router steps need their completed per-step artifacts because those CLIs are single-shot.

In [ ]:
for filename in ("train.npz", "validation.npz", "metadata.json"):
    assert (ROOT / "features" / filename).is_file(), "Complete feature extraction before backup."
assert json.loads((ROOT / "features/metadata.json").read_text())["source"] == SOURCE

for last in ROOT.rglob("last.pt"):
    check_checkpoint_pair(last.parent)
artifact_path = Path(kagglehub.dataset_download(HANDLE, force_download=True))
if not any(p.is_file() and p.name != ".complete" for p in artifact_path.rglob("*")):
    raise RuntimeError("Empty central artifact dataset; refusing to replace it.")
staging = ROOT.parent / (ROOT.name + "_artifact_sync")
if staging.exists():
    shutil.rmtree(staging)
shutil.copytree(artifact_path, staging)
destination = staging / EXPERIMENT
if destination.exists():
    shutil.rmtree(destination)
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(ROOT, destination)
kagglehub.dataset_upload(HANDLE, str(staging), ignore_patterns=[".complete"],
                         version_notes=f"Available research artifacts: {EXPERIMENT}")
print("Backed up:", EXPERIMENT)